# Smoking Prediction — Simple Augmented Model

This notebook is a **cleaner version** of the project.

### What it does
1. Loads the competition `train.csv` and `test.csv`.
2. Loads the extra real-world `smoking.csv`.
3. Removes duplicate rows from the extra data.
4. Removes any extra rows that match the competition train/test features (to avoid leakage).
5. Creates a few useful health-related features.
6. Trains **XGBoost** and **LightGBM**.
7. Combines them using an **80% XGBoost + 20% LightGBM** blend.
8. Creates the final `submission_xgb_lgb_80_20.csv`.

The target is `smoking`, and the submission keeps **probabilities** rather than converting them to 0/1 because ROC-AUC works better with probabilities.

## 1. Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("Libraries loaded!")

Libraries loaded!


## 2. Load the three datasets

In [ ]:
def find_file(names):
    for name in names:
        if os.path.exists(name):
            return name
    raise FileNotFoundError(f"Could not find any of: {names}")

train_file = find_file(["train.csv", "train (1).csv"])
test_file = find_file(["test.csv", "test (1).csv"])
extra_file = find_file(["smoking.csv"])

train = pd.read_csv(train_file)
test = pd.read_csv(test_file)
extra = pd.read_csv(extra_file)

print("Train:", train.shape)
print("Test :", test.shape)
print("Extra:", extra.shape)

Train: (15000, 24)
Test : (10000, 23)
Extra: (55692, 23)


## 3. Check the data

In [ ]:
print("Train columns:")
print(train.columns.tolist())

print("\nMissing values in train:")
print(train.isna().sum().sum())

print("\nSmoking rate:")
print(train["smoking"].mean())

Train columns:
['id', 'age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'dental caries', 'smoking']

Missing values in train:
0

Smoking rate:
0.3658


## 4. Safely add the real-world data

We do **not** blindly append the extra CSV.

We remove duplicate feature rows and also remove rows whose features already occur in the competition train or test set. This is important because using test-set labels would create target leakage.

In [ ]:
target = "smoking"
id_col = "id"
feature_cols = [c for c in test.columns if c != id_col]

def make_key(df):
    return pd.util.hash_pandas_object(df[feature_cols], index=False)

train_keys = set(make_key(train))
test_keys = set(make_key(test))

# Remove duplicate feature rows inside the extra data
extra_clean = extra.drop_duplicates(subset=feature_cols).copy()

# Remove anything matching competition train/test
extra_clean = extra_clean[
    ~make_key(extra_clean).isin(train_keys | test_keys)
].copy()

print("Original extra rows:", len(extra))
print("Unique extra rows  :", len(extra.drop_duplicates(subset=feature_cols)))
print("Rows actually added:", len(extra_clean))

augmented = pd.concat([train, extra_clean], ignore_index=True)

print("Final training rows:", len(augmented))

Original extra rows: 55692
Unique extra rows  : 44552
Rows actually added: 44552
Final training rows: 59552


## 5. Create simple useful features

In [ ]:
def make_features(df):
    x = df.drop(columns=[id_col, target], errors="ignore").copy()

    # Body measurements
    x["BMI"] = x["weight(kg)"] / ((x["height(cm)"] / 100) ** 2)
    x["waist_height_ratio"] = x["waist(cm)"] / x["height(cm)"]
    x["weight_height_ratio"] = x["weight(kg)"] / x["height(cm)"]

    # Blood pressure
    x["pulse_pressure"] = x["systolic"] - x["relaxation"]
    x["bp_product"] = x["systolic"] * x["relaxation"]

    # Cholesterol / blood markers
    x["chol_hdl_ratio"] = x["Cholesterol"] / (x["HDL"] + 0.001)
    x["ldl_hdl_ratio"] = x["LDL"] / (x["HDL"] + 0.001)
    x["trig_hdl_ratio"] = x["triglyceride"] / (x["HDL"] + 0.001)

    # Liver-related markers
    x["ast_alt_ratio"] = x["AST"] / (x["ALT"] + 0.001)
    x["alt_gtp_ratio"] = x["ALT"] / (x["Gtp"] + 0.001)
    x["liver_score"] = x["AST"] + x["ALT"] + x["Gtp"]

    return x.replace([np.inf, -np.inf], np.nan).fillna(0)

X = make_features(augmented)
y = augmented[target]

X_test = make_features(test)

print("Number of model features:", X.shape[1])

Number of model features: 33


## 6. Do a quick validation check

This is only to make sure the approach is sensible before producing the final submission.

A higher ROC-AUC is better.

In [ ]:
# Use only the original competition train rows for validation.
# The extra real-world data remains available for training.

train_part, valid_part = train_test_split(
    train,
    test_size=0.20,
    random_state=42,
    stratify=train[target]
)

validation_training = pd.concat([train_part, extra_clean], ignore_index=True)

X_train = make_features(validation_training)
y_train = validation_training[target]

X_valid = make_features(valid_part)
y_valid = valid_part[target]

xgb_check = XGBClassifier(
    n_estimators=900,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=3,
    min_child_weight=3,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_check.fit(X_train, y_train)
valid_prediction = xgb_check.predict_proba(X_valid)[:, 1]

print("Validation ROC-AUC:", roc_auc_score(y_valid, valid_prediction))

Validation ROC-AUC: 0.8970315545525921


## 7. Train the final XGBoost model

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=900,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=3,
    min_child_weight=3,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_model.fit(X, y)

xgb_prediction = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost predictions created!")

XGBoost predictions created!


## 8. Train a second model: LightGBM

In [ ]:
lgb_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

lgb_model.fit(X, y)

lgb_prediction = lgb_model.predict_proba(X_test)[:, 1]

print("LightGBM predictions created!")

LightGBM predictions created!


## 9. Make the final prediction

We give XGBoost more weight because it performed better in our validation test.

**80% XGBoost + 20% LightGBM**.

In [ ]:
final_prediction = (
    0.80 * xgb_prediction
    + 0.20 * lgb_prediction
)

submission = pd.DataFrame({
    "id": test["id"],
    "smoking": final_prediction
})

submission.to_csv("submission_xgb_lgb_80_20.csv", index=False)

print(submission.head())
print("\nSubmission shape:", submission.shape)
print("Saved as: submission_xgb_lgb_80_20.csv")

      id   smoking
0  15000  0.776884
1  15001  0.507068
2  15002  0.452547
3  15003  0.674932
4  15004  0.591104

Submission shape: (10000, 2)
Saved as: submission_xgb_lgb_80_20.csv


## 10. Final checks

Before submitting, make sure:

- There are exactly 10,000 rows.
- The columns are `id` and `smoking`.
- `smoking` contains probabilities between 0 and 1.
- You submit the CSV created in the previous cell.

In [ ]:
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
print("Min prediction:", submission["smoking"].min())
print("Max prediction:", submission["smoking"].max())

assert len(submission) == len(test)
assert submission.columns.tolist() == ["id", "smoking"]
assert submission["smoking"].between(0, 1).all()

print("\nEverything looks ready!")

Rows: 10000
Columns: ['id', 'smoking']
Min prediction: 0.0020582182325433267
Max prediction: 0.9576433739266473

Everything looks ready!
